Check if newly generated reduced endo A model has all the necessary exchange rxs​

List of missing boundary reactions: model_full_THG_missing_exchange.txt​

If not present, add them manually

In [3]:
import sys
import cobra
import os
import pdb

def read_list(path):
    """
    Reads a text file where fields are separated by tabs.
    Skips the first line (header).
    Returns a list of lists, where each inner list represents a row's fields
    (ec_reaction, ec_met, thg_met).
    """
    
    #open the file and separate the fields by tabs. Skip the first line (header).
    with open(path, 'r', encoding='utf-8') as file: # Added encoding for better compatibility
        lines = file.readlines()[1:] # Skip the header
    
    # Split each line by tab character and strip whitespace
    return [line.strip().split('\t') for line in lines if line.strip()]


def main():
    crucial_exch_rxns = "model_full_THG_missing_exchange_edited.txt"
    crucial_exch_rxns="ec_met_tsv.tsv"
    thg_model_path = os.path.join("transcriptomics", "model_THG_endoA_reduced_3105.xml")
    full_thg_model_path = "model_full_THG_round2.xml"
    ec_model_path = "EC_model_with_KEGG.xml"
    reinis_model_path = "THG-beta2_endoB_reinis.xml"

    thg_model = cobra.io.read_sbml_model(thg_model_path)
    ec_model = cobra.io.read_sbml_model(ec_model_path)
    full_thg_model = cobra.io.read_sbml_model(full_thg_model_path)
    reinis_model = cobra.io.read_sbml_model(reinis_model_path)
    rxn_list = read_list(crucial_exch_rxns)
    ec_rxn_id, ec_met, thg_met = zip(*rxn_list)

    #clean up thg_met lists (keep only first word)

    thg_met = [met.split()[0] for met in thg_met]

    print("Number of crucial exchange reactions:", len(ec_rxn_id))
    found_rs= 0
    not_found_rs = 0

    mets_missing_exchanges = list() #Metabolites in the list that are in our reduced THG model but do not have any exchange reactions associated with them
    mets_in_no_thg = list() #Metabolites in the list that are not found in the reduced THG model, full THG model or Reinis model 
    mets_full_thg = list() #Metabolites in the list that are not found in the reduced THG model but are found in the full THG model
    mets_in_old_thg = list() #Metabolites in the list that are not found in the reduced THG model but are found in the older THG model (Reinis model)

    #Sublists, grouped whether the metabolite has associated exchange reactions or not
    mets_full_thg_with_exchanges = list() #Metabolites in the full THG model that have associated exchange reactions
    mets_full_thg_without_exchanges = list() #Metabolites in the full THG model that do not have associated exchange reactions
    exchange_rs_full_thg = list() #Exchange reactions in the full THG model that are associated with metabolites in the list
    mets_in_old_thg_with_exchanges = list() #Metabolites in the old THG model that have associated exchange reactions
    mets_in_old_thg_without_exchanges = list() #Metabolites in the old THG model that do not have associated exchange reactions

    found_exchanges = list() #List of metabolites that have associated exchange reactions in the reduced THG model

    for i, thg_met_id in enumerate(thg_met):
        try:
            met = thg_model.metabolites.get_by_id(thg_met_id)
            
        except KeyError: #Metabolite not found in reduced THG model
            not_found_rs += 1

            try: #See if metabolite is present in full THG model
                met = full_thg_model.metabolites.get_by_id(thg_met_id)
                print(f"{thg_met_id} not found in reduced THG but found in full THG. Metabolite name: {met.name}")
                mets_full_thg.append(thg_met_id)
                #check if metabolite in full THG model is associated with any boundary reactions
                rxs = [rxn for rxn in full_thg_model.boundary if met in rxn.metabolites]
                if rxs:
                    print(f"Boundary associated with {thg_met_id} in Full THG model:")
                    mets_full_thg_with_exchanges.append(thg_met_id)
                    for rxn in rxs:
                        print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction}")
                        exchange_rs_full_thg.append(rxn.id)

                else:
                    mets_full_thg_without_exchanges.append(thg_met_id)
                    print(f"No boundary reactions found for {thg_met_id} in Full THG model.")

            except KeyError: #Metabolite not found in full THG model, check if it is in the older THG model (Reinis model)
                print(f"{thg_met_id} not found in the THG model metabolites (nor full nor reduced THG). ")

                try:
                    met = reinis_model.metabolites.get_by_id(thg_met_id)
                    print(f"{thg_met_id} found in Reinis model. Metabolite name: {met.name}")
                    mets_in_old_thg.append(thg_met_id)

                    #check if metabolite is associated with any boundary reactions in Reinis model
                    rxs = [rxn for rxn in reinis_model.boundary if met in rxn.metabolites]
                    if rxs:
                        print(f"Boundary reactions associated with {thg_met_id} in Reinis model:")
                        mets_in_old_thg_with_exchanges.append(thg_met_id)
                        for rxn in rxs:
                            print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction}")
                    else:
                        mets_in_old_thg_without_exchanges.append(thg_met_id)
                        print(f"No boundary reactions found for {thg_met_id} in Reinis model.")

                except KeyError: #Metabolite not found in Reinis model either

                    try:
                        met = ec_model.metabolites.get_by_id(ec_met[i])
                        print("Metabolite found in EC model: ", met.id)
                        mets_in_no_thg.append(thg_met_id)
                    except KeyError:
                        #try to retrieve the reaction instead
                        try:
                            ec_rxn = ec_model.reactions.get_by_id(ec_rxn_id[i])
                            print(f"{ec_met[i]} not found in the EC model metabolites. Found in EC reaction: {ec_rxn.id} | {ec_rxn.reaction}")
                        except KeyError:
                            print(f"{ec_met[i]} not found in the EC model metabolites and reactions. Skipping...")
                    continue
            continue

        # Find exchange reactions in THG model involving this metabolite
        associated_exchanges = [
            rxn for rxn in thg_model.exchanges if met in rxn.metabolites
        ]

        if associated_exchanges:
            found_rs += 1
            for rxn in associated_exchanges:
                found_exchanges.append(thg_met_id)
            # print(f"\nExchange reactions for {thg_met_id}: {[rxn.id for rxn in associated_exchanges]}")
            # try:
            #     ec_rxn = ec_model.reactions.get_by_id(ec_rxn_id[i])
            #     print(f"EC exchange reaction: {ec_rxn_id[i]}, reaction: {ec_rxn.reaction}")
            # except KeyError:
            #     print(f"EC reaction {ec_rxn_id[i]} not found in EC model.")
            # print("In THG model:")
            # for rxn in associated_exchanges:
            #     print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction}")
        else:
            not_found_rs += 1
            # print(f"No exchange reactions found for {thg_met_id}")
            mets_missing_exchanges.append(thg_met_id)

    print(f"\nTotal found exchange reactions: {found_rs}")
    print(f"Total not found exchange reactions: {not_found_rs}")

    print(f"\nMetabolites in old THG model but not in new: {mets_in_old_thg}")
    print(f"Metabolites not found in new THG or older THG model: {mets_in_no_thg}")
    print(f"Metabolites missing exchange reactions in new THG model: {mets_missing_exchanges}")

    print(f"\nMetabolites in full THG model with exchanges: {mets_full_thg_with_exchanges}")
    print(f"Metabolites in full THG model without exchanges: {mets_full_thg_without_exchanges}")
    print(f"Metabolites in old THG model with exchanges: {mets_in_old_thg_with_exchanges}")
    print(f"Metabolites in old THG model without exchanges: {mets_in_old_thg_without_exchanges}")

    print(f"\nFound exchanges in reduced THG model: {found_exchanges}")
    print(f"Exchange reactions in full THG model associated with metabolites in the list that are not in reduced model: {exchange_rs_full_thg}")

    pdb.set_trace()  # Set a breakpoint for debugging if needed

if __name__ == "__main__":
    main()


Number of crucial exchange reactions: 75
MAM03758c not found in reduced THG but found in full THG. Metabolite name: Melanin
No boundary reactions found for MAM03758c in Full THG model.
MAM01665e not found in reduced THG but found in full THG. Metabolite name: dem2emgacpail_prot heparan sulfate
Boundary associated with MAM01665e in Full THG model:
  reaction ID: MAR09705 | reaction: MAM01665e --> 
MAM01687e not found in reduced THG but found in full THG. Metabolite name: dgpi_prot heparan sulfate
Boundary associated with MAM01687e in Full THG model:
  reaction ID: MAR09704 | reaction: MAM01687e --> 
MAM03591c not found in reduced THG but found in full THG. Metabolite name: Glucose-1,3-Mannose Oligosaccharide
No boundary reactions found for MAM03591c in Full THG model.
MAM03593c not found in reduced THG but found in full THG. Metabolite name: Glucose-1,2-(2)[Glucose-1,3]-Mannose Oligosaccharide
No boundary reactions found for MAM03593c in Full THG model.
MAM01955e not found in reduced TH

In [6]:
full_thg_model_path = "model_full_THG_round2.xml"
ec_model_path = "EC_model_with_KEGG.xml"

full_thg_model = cobra.io.read_sbml_model(full_thg_model_path)
ec_model = cobra.io.read_sbml_model(ec_model_path)

ec_mets = ["n5m2masn[g]", "clpnd[e]"]
thg_mets = ["MAM02516g", "MAM01741e"]

#check if metabolites in thg list have a boundary reaction in the full THG model

for i, thg_met in enumerate(thg_mets):
    try:
        met = full_thg_model.metabolites.get_by_id(thg_met)
        print(f"{thg_met} found in full THG model: {met.name}")
        
        # Check if the metabolite is associated with any boundary reactions
        rxs = [rxn for rxn in full_thg_model.boundary if met in rxn.metabolites]
        if rxs:
            print(f"Boundary reactions associated with {thg_met}:")
            for rxn in rxs:
                print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction} | Bounds: {rxn.lower_bound}, {rxn.upper_bound}")

            try:
                ec_met = ec_model.metabolites.get_by_id(ec_mets[i])
                
                rxs_ec = [rxn for rxn in ec_model.boundary if ec_met in rxn.metabolites]
                if rxs_ec:
                    print(f"Boundary reactions associated with {ec_mets[i]} in EC model:")
                    for rxn in rxs_ec:
                        print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction} | Bounds: {rxn.lower_bound}, {rxn.upper_bound}")
                else:
                    print(f"No boundary reactions found for {ec_mets[i]} in EC model.")

            except KeyError:
                print(f"{ec_mets[i]} not found in EC model.")


        else:
            print(f"No boundary reactions found for {thg_met} in Full THG model.")

            # Check if metabolite in ec model has boundary reactions and print them and boundary reactions
            try:
                ec_met = ec_model.metabolites.get_by_id(ec_mets[i])
                
                rxs_ec = [rxn for rxn in ec_model.boundary if ec_met in rxn.metabolites]
                if rxs_ec:
                    print(f"Boundary reactions associated with {ec_mets[i]} in EC model:")
                    for rxn in rxs_ec:
                        print(f"  reaction ID: {rxn.id} | reaction: {rxn.reaction} | Bounds: {rxn.lower_bound}, {rxn.upper_bound}")
                else:
                    print(f"No boundary reactions found for {ec_mets[i]} in EC model.")
            
            except KeyError:
                print(f"{ec_mets[i]} not found in EC model.")

                

    
    except KeyError:
        print(f"{thg_met} not found in full THG model.")



MAM02516g found in full THG model: n5m2masn
No boundary reactions found for MAM02516g in Full THG model.
Boundary reactions associated with n5m2masn[g] in EC model:
  reaction ID: DM_n5m2masn_g | reaction: n5m2masn[g] -->  | Bounds: 0.0, 0.0
MAM01741e found in full THG model: DPA
Boundary reactions associated with MAM01741e:
  reaction ID: MAR00571 | reaction: MAM01741e <=>  | Bounds: -1000.0, 1000.0
Boundary reactions associated with clpnd[e] in EC model:
  reaction ID: EX_clpnd[e] | reaction: clpnd[e] <=>  | Bounds: -1000.0, 1000.0


Code to generate table of metabolites:

In [10]:
import cobra
import pandas as pd

# 1) Load your model
full_thg_model_path = "model_full_THG_round2.xml"
model = cobra.io.read_sbml_model(full_thg_model_path)

# 2) Define the non-canonical compartments
comp_codes = ['ca','cb','cj','ci','ck','a','y','v']  # two-letter first

# 3) Filter metabolites in those compartments
mets = [m for m in model.metabolites if m.compartment in comp_codes]

# 4) Compute "core" IDs and collect versions
core_map = {}
for m in mets:
    mid = m.id
    core = mid
    for code in comp_codes:
        if mid.endswith(code):
            core = mid[:-len(code)]
            break
    entry = core_map.setdefault(core, {'versions': [], 'met_objs': []})
    entry['versions'].append(mid)
    entry['met_objs'].append(m)

# 5) Build DataFrame rows, safely handling missing annotation keys
rows = []
for core, info in core_map.items():
    versions = sorted(set(info['versions']))
    m0 = info['met_objs'][0]
    ann = m0.annotation or {}
    # helper to flatten lists or missing entries
    def norm_list(key):
        val = ann.get(key, [])
        if isinstance(val, list):
            return "|".join(val)
        elif val is None:
            return ""
        else:
            return str(val)
    rows.append({
        'core_id':           core,
        'versions':          ";".join(versions),
        'kegg.compound':     norm_list('kegg.compound'),
        'pubchem.compound':  norm_list('pubchem.compound'),
        'hmdb':              norm_list('hmdb'),
        'inchi':             ann.get('inchi', '') or '',
        'name':              m0.name or '',
        'formula':           m0.formula or ''
    })

df = pd.DataFrame(rows, 
                  columns=[
                      'core_id','versions',
                      'kegg.compound','pubchem.compound',
                      'hmdb','inchi','name','formula'
                  ])

# 6) Preview & save
print(df.head(10).to_markdown(index=False))
excel_path = "noncanonical_mets_EndoA.xlsx"
df.to_excel(excel_path, index=False)
print(f"Saved full table with {len(df)} rows to {excel_path}")


| core_id   | versions                                                                             | kegg.compound   |   pubchem.compound | hmdb      | inchi                                                                                                                                                                                                                                                                                                                   | name                    | formula       |
|:----------|:-------------------------------------------------------------------------------------|:----------------|-------------------:|:----------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:------------------------|:

Compare model before and after adding demand rs

In [1]:
import cobra
import pandas as pd

#before demand reactions were added
model_before_path ="model_full_THG_round2.xml"
model_before = cobra.io.read_sbml_model(model_before_path)
#after demand reactions were added
model_after_path = "model_after_round1_glycocalix_cytoskeleton.xml"
model_after = cobra.io.read_sbml_model(model_after_path)

#compare number of reactions in both models
print(f"Number of reactions in model before: {len(model_before.reactions)}")
print(f"Number of reactions in model after: {len(model_after.reactions)}")

#compare number of demand reactions in both models
demand_reactions_before = [rxn for rxn in model_before.reactions if rxn.id.startswith("DM_")]
print(f"Number of demand reactions in model before: {len(demand_reactions_before)}")
demand_reactions_after = [rxn for rxn in model_after.reactions if rxn.id.startswith("DM_")]
print(f"Number of demand reactions in model after: {len(demand_reactions_after)}")



Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-22
Number of reactions in model before: 24388
Number of reactions in model after: 24532
Number of demand reactions in model before: 0
Number of demand reactions in model after: 144


Add missing exchange reactions

In [4]:
import re
import cobra
import pandas as pd

def get_mar_id_generator(model, start_default=90000):
    """
    Yields unique MAR##### IDs, starting one above the current max in `model`.
    """
    pat = re.compile(r"MAR(\d{5})$")
    nums = [
        int(pat.match(rxn.id).group(1))
        for rxn in model.reactions
        if pat.match(rxn.id)
    ]
    current = max(nums) if nums else start_default
    while True:
        current += 1
        yield f"MAR{current:05d}"

def classify_reaction(rxn, model):
    """Return 'exchange', 'demands', 'sink', or 'internal'."""
    if rxn in model.exchanges:
        return 'exchange'
    if rxn in model.demands:
        return 'demand'
    if rxn in model.sinks:
        return 'sink'
    return 'internal'

def add_missing_exchanges(full_thg, ec_model, cases):
    created = []
    mar_gen = get_mar_id_generator(full_thg)

    for case in cases:
        ec_rxn = ec_model.reactions.get_by_id(case['ec_rxn'])
        thg_met = full_thg.metabolites.get_by_id(case['thg_met'])

        # 1) classify the EC reaction
        rxn_type = classify_reaction(ec_rxn, ec_model)
        print(f"{ec_rxn.id} → {rxn_type}")

        # 2) pull out the single-metabolite + coeff
        mets = list(ec_rxn.metabolites.items())
        if len(mets) != 1:
            raise ValueError(
                f"Reaction {ec_rxn.id} contains {len(mets)} metabolites; "
                "cannot auto-infer which to use."
            )
        ec_met_obj, stoich = mets[0]

        # 3) build your new MAR##### reaction
        new_id = next(mar_gen)
        new_rxn = cobra.Reaction(new_id)
        new_rxn.name        = ec_rxn.name
        new_rxn.lower_bound = ec_rxn.lower_bound
        new_rxn.upper_bound = ec_rxn.upper_bound
        new_rxn.add_metabolites({thg_met: stoich})
        # optional: record what EC met we used
        new_rxn.notes['ec_met_used'] = ec_met_obj.id
        new_rxn.notes['type']        = rxn_type

        full_thg.add_reactions([new_rxn])
        #check if model is feasible
        feasible = full_thg.optimize().status == 'optimal'
        if not feasible:
            print(f"Warning: Adding {new_id} made the model infeasible!. Reaction: {new_rxn.reaction} | Type: {rxn_type} | EC: {ec_rxn.id} | THG Met: {thg_met.id}")
            full_thg.remove_reactions([new_rxn])
            continue
        created.append(new_id)

    return created

def main():
    # --- load models --------------------------------------------------------
    full_thg = cobra.io.read_sbml_model("model_after_round1_glycocalix_cytoskeleton.xml")
    ec      = cobra.io.read_sbml_model("EC_model_with_KEGG.xml")

    # --- read your 34-case table from Excel — drop ec_met entirely --------
    df = pd.read_excel("missing_exchanges_endoA.xlsx")
    required = {'ec_rxn','thg_met','reason'}
    if not required.issubset(df.columns):
        missing = required - set(df.columns)
        raise ValueError(f"Excel file is missing columns: {missing}")

    cases = df[list(required)].to_dict('records')

    # --- create the new boundary reactions ---------------------------------
    created = add_missing_exchanges(full_thg, ec, cases)
    print(f"\nAdded {len(created)} reactions:")
    for rxn_id in created:
        print("  ", rxn_id)

    # --- save outputs -------------------------------------------------------
    
    with open("extra_boundary_reactions.txt","w") as f:
        f.write("\n".join(created))

    cobra.io.write_sbml_model(full_thg, "model_full_THG_with_exchanges.xml")
if __name__ == "__main__":
    main()


DM_dctp(m) → demand
DM_dgtp(m) → demand
DM_dttp(m) → demand
DM_gpi_sig(er) → demand
DM_kdn_c → demand
sink_pre_prot(r) → internal
DM1p2cbxl → demand
Dmidour → demand
EXcore2 → demand
Excspg_b → demand
Excspg_c → demand
Excspg_d → demand
EX5MTP → exchange
EX_15HETE → exchange
EX_12(13)DHOME → exchange


c:\Users\MFRA0106\AppData\Local\anaconda3\Lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


EX_13oODE → exchange
EX_c4crn → exchange
EX_5oxoe → exchange
EX_dhcrm → exchange
EX_12HETE → exchange
EX_3h5chola → exchange
EX_odecrn[e] → exchange
EX_ttdcrn[e] → exchange
EX_5eipenc[e] → exchange
DM_melanin_c → demand
DM_glc1man_g → demand
DM_glc3man_g → demand
DM_mem2emgacpail_prot_hs_r → demand
EX_13HODE → exchange
EX_lkynre → exchange
sink_Ser-Gly/Ala-X-Gly[r] → internal
EX_hdcea[e] → exchange
EX_tmao[e] → exchange
DM_n5m2masn_g → demand

Added 33 reactions:
   MAR24007
   MAR24008
   MAR24009
   MAR24010
   MAR24011
   MAR24012
   MAR24013
   MAR24014
   MAR24015
   MAR24016
   MAR24017
   MAR24018
   MAR24019
   MAR24020
   MAR24022
   MAR24023
   MAR24024
   MAR24025
   MAR24026
   MAR24027
   MAR24028
   MAR24029
   MAR24030
   MAR24031
   MAR24032
   MAR24033
   MAR24034
   MAR24035
   MAR24036
   MAR24037
   MAR24038
   MAR24039
   MAR24040


In [10]:
import re
import cobra
import pandas as pd

# model_path = "model_full_THG_with_exchanges.xml"
# model = cobra.io.read_sbml_model(model_path)

rx = model.reactions.get_by_id("MAR24038")
# Check if the reaction is a boundary reaction
if rx in model.exchanges:
    print(f"{rx.id} is an exchange reaction.")

print(f"Reaction ID: {rx.id}")
print(f"Reaction Name: {rx.name}")  
print(f"Reaction Equation: {rx.reaction}")
print(f"Lower Bound: {rx.lower_bound}")
print(f"Upper Bound: {rx.upper_bound}")
print(f"Metabolites: {rx.metabolites}")
print(f"Reaction: {rx.reaction}")
print(f"Notes: {rx.notes}")

MAR24038 is an exchange reaction.
Reaction ID: MAR24038
Reaction Name: EX_hdcea[e]
Reaction Equation: MAM02116e <=> 
Lower Bound: -1000.0
Upper Bound: 1000.0
Metabolites: {<Metabolite MAM02116e at 0x22059a06990>: -1.0}
Reaction: MAM02116e <=> 
Notes: {'ec_met_used': 'hdcea[e]', 'type': 'exchange'}


In [12]:
mets_to_find= ['MAM02926e', 'MAM01285e', 'MAM00010e', 'MAM01005e', 'MAM01342e', 'MAM02319e', 'MAM03107e', 'MAM01517e', 'MAM02147e', 'MAM01554e', 'MAM01965e', 'MAM02403e', 'MAM01628e', 'MAM02949e', 'MAM02634e', 'MAM02657e', 'MAM01370e', 'MAM01708e', 'MAM01306e', 'MAM01862e', 'MAM02439e', 'MAM01619e', 'MAM02409e', 'MAM02411e', 'MAM02808e', 'MAM01983e', 'MAM02914e', 'MAM02927e', 'MAM02738e', 'MAM02696e', 'MAM01197e', 'MAM02343e', 'MAM03706e', 'MAM02344e', 'MAM03488e'] 
num_rs=0
for met_id in mets_to_find:
    met = model.metabolites.get_by_id(met_id)
    #find boundary reactions associated with this metabolite
    boundary_reactions = [rxn for rxn in model.boundary if met in rxn.metabolites]
    if boundary_reactions:
        for rxn in boundary_reactions:
            print(rxn.id)
            num_rs += 1

print(f"Total number of boundary reactions found: {num_rs}")

MAR09715
MAR09255
MAR13042
MAR09010
MAR09028
MAR09857
MAR04382
MAR09117
MAR09386
MAR09121
MAR09034
MAR09135
MAR09065
MAR09088
MAR09920
MAR09921
MAR09070
MAR09848
MAR09259
MAR11400
MAR11404
MAR09290
MAR04918
MAR09925
MAR00661
MAR09085
MAR09868
MAR11959
MAR09845
MAR09858
MAR13052
MAR10256
MAR04943
MAR10434
MAR04859
Total number of boundary reactions found: 35


In [14]:
#open text file with reactions (protected_rs_endoA.txt) and check if they are in the model
with open("protected_rs_endoA.txt", "r") as file:
    protected_reactions = [line.strip() for line in file if line.strip()]

# Check if each protected reaction is in the model
for rxn_id in protected_reactions:
    try:
        rxn = model.reactions.get_by_id(rxn_id)
    except KeyError:
        print(f"{rxn_id} is NOT in the model.")


In [1]:
import cobra

model_path= "transcriptomics/model_THG_endoA_reduced_1606.xml"


model = cobra.io.read_sbml_model(model_path)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-22


In [3]:
#print all reactions in glycocalix ("y")

for rxn in model.reactions:
    for met in rxn.metabolites:
        if met.compartment == "y":
            print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}")

glycocalix_mets = [met for met in model.metabolites if met.compartment == "y"]
#do fva on all reactions in glycocalix
from cobra.flux_analysis import flux_variability_analysis
glycocalix_rxns = [rxn for rxn in model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]
fva_results = flux_variability_analysis(model, reaction_list=glycocalix_rxns, fraction_of_optimum=0.95)
# Print FVA results for glycocalix reactions
print("FVA results for glycocalix reactions:")
for rxn_id, (min_flux, max_flux) in fva_results.iterrows():
    print(f"Reaction ID: {rxn_id}, Min Flux: {min_flux}, Max Flux: {max_flux}")
    


Reaction ID: MARMAR49926, Reaction: MAM01371e <=> MAM01371y
Reaction ID: DM_MAM01371y, Reaction: MAM01371y --> 
FVA results for glycocalix reactions:
Reaction ID: MARMAR49926, Min Flux: 0.0, Max Flux: 1000.0
Reaction ID: DM_MAM01371y, Min Flux: 0.0, Max Flux: 1000.0


In [3]:
full_model_path = "model_full_THG_with_exchanges.xml"
full_model = cobra.io.read_sbml_model(full_model_path)


In [10]:
met = full_model.metabolites.get_by_id("MAM01371y")
print(met.reactions)
for rxn in met.reactions:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")

frozenset({<Reaction MAR22899 at 0x218974cac90>, <Reaction MARMAR49926 at 0x21897921850>, <Reaction DM_MAM01371y at 0x21897cef8d0>})
Reaction ID: MAR22899, Reaction: 2.0 MAM00193y + 4.0 MAM01371y --> MAM00192y + 4.0 MAM01285y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR49926, Reaction: MAM01371e <=> MAM01371y, Lower Bound: -1000.0, Upper Bound: 1000.0
Reaction ID: DM_MAM01371y, Reaction: MAM01371y --> , Lower Bound: 0.0, Upper Bound: 1000.0


In [13]:
met2= full_model.metabolites.get_by_id("MAM00193y")
print(met2.name)
print(met2.formula)
met2rxns = [rxn for rxn in full_model.reactions if met2 in rxn.metabolites]
for rxn in met2rxns:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")

[phosphorylase B]
X
Reaction ID: MAR22899, Reaction: 2.0 MAM00193y + 4.0 MAM01371y --> MAM00192y + 4.0 MAM01285y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: DM_MAM00193y, Reaction: MAM00193y --> , Lower Bound: 0.0, Upper Bound: 1000.0


In [14]:
met3= full_model.metabolites.get_by_id("MAM00192y")
print(met3.name)
print(met3.formula)
met3rxns = [rxn for rxn in full_model.reactions if met3 in rxn.metabolites]
for rxn in met3rxns:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")

[phosphorylase A]
O12P4X2
Reaction ID: MAR22899, Reaction: 2.0 MAM00193y + 4.0 MAM01371y --> MAM00192y + 4.0 MAM01285y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: DM_MAM00192y, Reaction: MAM00192y --> , Lower Bound: 0.0, Upper Bound: 1000.0


In [15]:
met4= full_model.metabolites.get_by_id("MAM01285y")
print(met4.name)
print(met4.formula)
met4rxns = [rxn for rxn in full_model.reactions if met4 in rxn.metabolites]
for rxn in met4rxns:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")

ADP
C10H12N5O10P2
Reaction ID: MAR22899, Reaction: 2.0 MAM00193y + 4.0 MAM01371y --> MAM00192y + 4.0 MAM01285y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: DM_MAM01285y, Reaction: MAM01285y --> , Lower Bound: 0.0, Upper Bound: 1000.0


In [16]:
#print all reactions in glycocalix that do not start with DM

glycocalix_rxns = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites) and not rxn.id.startswith("DM_")]
for rxn in glycocalix_rxns:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")

Reaction ID: MAR13600, Reaction: 2.0 MAM02497y + MAM02555y + 2.0 MAM02630y --> 2.0 MAM01588y + MAM02039y + 2.0 MAM02040y + MAM02554y + 2.0 MAM02609y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MAR13608, Reaction: MAM01365y + MAM02039y + MAM02555y + MAM02630y <=> MAM02040y + MAM02497y + MAM02554y, Lower Bound: -1000.0, Upper Bound: 1000.0
Reaction ID: MAR16365, Reaction: MAM01524y + 2.0 MAM02040y --> MAM01520y + MAM02039y + MAM02525y + MAM02946y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MAR16372, Reaction: MAM01522y + MAM02040y --> MAM01523y + MAM02525y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MAR16377, Reaction: MAM01529y + 2.0 MAM02040y --> MAM01527y + MAM02039y + MAM02525y + MAM02946y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MAR16419, Reaction: MAM01559y + MAM02040y --> MAM01560y + MAM02039y + MAM02946y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MAR21784, Reaction: MAM01562y + 3.0 MAM02040y --> MAM01557y + 2.0 MAM02039y + MAM02525y + 2

In [18]:
met5= full_model.metabolites.get_by_id("MAM02039y")
print(met5.name)

H+


Adding all sink reactions:

In [ ]:
#list of metabolites to add sink reactions for
import cobra
from cobra.flux_analysis import flux_variability_analysis
full_model_path = "model_full_THG_with_exchanges.xml"
full_model = cobra.io.read_sbml_model(full_model_path)

mets_to_add_sinks = ["MAM01559y", "MAM01562y", "MAM01524y", "MAM01522y", "MAM00193y", "MAM01529y", "MAM02555y", "MAM02630y","MAM01365y"]
for met_id in mets_to_add_sinks:
    print(f"Name of the metabolite: {full_model.metabolites.get_by_id(met_id).name}")

sink_rxn_list = []  # List to keep track of added sink reactions

# Add sink reactions for each metabolite in the list
for met_id in mets_to_add_sinks:
    try:
        met = full_model.metabolites.get_by_id(met_id)
        # Create a new sink reaction
        sink_rxn = cobra.Reaction(f"Sink_{met.id}")
        sink_rxn.name = f"Sink {met.name}"
        sink_rxn.subsystem = 'Putative Sink Reaction'
        sink_rxn.lower_bound = -1000  # Set lower bound to 0
        sink_rxn.upper_bound = 1000  # Set upper bound to a large value
        sink_rxn.add_metabolites({met: 1})  # Add the metabolite as a product
        sink_rxn.notes = {'Confidence Level': '4', 'AUTHORS': 'Sink transport reaction automatically generated'}
        sink_rxn.gene_reaction_rule = ''  # No gene association for sink reactions
        # Add the sink reaction to the model
        full_model.add_reactions([sink_rxn])
        sink_rxn_list.append(sink_rxn.id)  # Keep track of added sink reactions
        print(f"Added sink reaction: {sink_rxn.id} for metabolite: {met.name}")

    except KeyError:
        print(f"Metabolite {met_id} not found in the model.")

# do an FVA on the model after adding sink reactions (only for glycocalix reactions)
glycocalix_rxns_after = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]
fva_results_after = flux_variability_analysis(full_model, reaction_list=glycocalix_rxns_after, fraction_of_optimum=0.95)
# Print FVA results for glycocalix reactions after adding sink reactions    
for rxn_id, row in fva_results_after.iterrows():
    min_flux = row['minimum']
    max_flux = row['maximum']
    print(f"Reaction ID: {rxn_id}, Min Flux: {min_flux}, Max Flux: {max_flux}")




Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-22
Name of the metabolite: chondroitin sulfate E (GalNAc4,6diS-GlcA), degradation product 5
Name of the metabolite: chondroitin sulfate E (GalNAc4,6diS-GlcA), free chain
Name of the metabolite: chondroitin sulfate A (GalNAc4S-GlcA), free chain
Name of the metabolite: chondroitin sulfate A (GalNAc4S-GlcA), degradation product 4
Name of the metabolite: [phosphorylase B]
Name of the metabolite: chondroitin sulfate B - dermatan sulfate (IdoA2S-GalNAc4S), free chain
Name of the metabolite: NADPH
Name of the metabolite: O2
Name of the metabolite: L-Arginine
Added sink reaction: Sink_MAM01559y for metabolite: chondroitin sulfate E (GalNAc4,6diS-GlcA), degradation product 5
Removed DM reaction: DM_MAM01559y for metabolite: chondroitin sulfate E (GalNAc4,6diS-GlcA), degradation product 5
Added sink reaction: Sink_MAM01562y for metabolite: chondroitin sulfate E (GalNAc4,6diS-GlcA), free chain
Removed DM react

Adding all sink reactions and removing demand rs:

In [ ]:
#list of metabolites to add sink reactions for
import cobra
from cobra.flux_analysis import flux_variability_analysis
full_model_path = "model_full_THG_with_exchanges.xml"
full_model = cobra.io.read_sbml_model(full_model_path)

mets_to_add_sinks = ["MAM01559y", "MAM01562y", "MAM01524y", "MAM01522y", "MAM00193y", "MAM01529y", "MAM02555y", "MAM02630y","MAM01365y"]
for met_id in mets_to_add_sinks:
    print(f"Name of the metabolite: {full_model.metabolites.get_by_id(met_id).name}")

sink_rxn_list = []  # List to keep track of added sink reactions

# Add sink reactions for each metabolite in the list
for met_id in mets_to_add_sinks:
    try:
        met = full_model.metabolites.get_by_id(met_id)
        # Create a new sink reaction
        sink_rxn = cobra.Reaction(f"Sink_{met.id}")
        sink_rxn.name = f"Sink {met.name}"
        sink_rxn.subsystem = 'Putative Sink Reaction'
        sink_rxn.lower_bound = -1000  # Set lower bound to 0
        sink_rxn.upper_bound = 1000  # Set upper bound to a large value
        sink_rxn.add_metabolites({met: 1})  # Add the metabolite as a product
        sink_rxn.notes = {'Confidence Level': '4', 'AUTHORS': 'Sink transport reaction automatically generated'}
        sink_rxn.gene_reaction_rule = ''  # No gene association for sink reactions
        # Add the sink reaction to the model
        full_model.add_reactions([sink_rxn])
        sink_rxn_list.append(sink_rxn.id)  # Keep track of added sink reactions
        print(f"Added sink reaction: {sink_rxn.id} for metabolite: {met.name}")

        #remove DM reaction if it exists
        for rxn in met.reactions:
            if rxn.id.startswith("DM_"):
                full_model.remove_reactions([rxn])
                print(f"Removed DM reaction: {rxn.id} for metabolite: {met.name}")
    except KeyError:
        print(f"Metabolite {met_id} not found in the model.")

# do an FVA on the model after adding sink reactions (only for glycocalix reactions)
glycocalix_rxns_after = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]
fva_results_after = flux_variability_analysis(full_model, reaction_list=glycocalix_rxns_after, fraction_of_optimum=0.95)
# Print FVA results for glycocalix reactions after adding sink reactions    
for rxn_id, row in fva_results_after.iterrows():
    min_flux = row['minimum']
    max_flux = row['maximum']
    print(f"Reaction ID: {rxn_id}, Min Flux: {min_flux}, Max Flux: {max_flux}")




Now trying to add transport rs instead of sink rs:

In [ ]:
model_transport = cobra.io.read_sbml_model("model_transport_allrxns_glycocalix_cytoskeleton.xml")

#see reactions in glycocalix (y)

glycocalix_rxns_transport = [rxn for rxn in model_transport.reactions if any(met.compartment == "y" for met in rxn.metabolites)]
for rxn in glycocalix_rxns_transport:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")
    #change lower bound to-1000 to make the reaction reversible
    rxn.lower_bound = -1000

# Print the reactions after changing the lower bound to check the changes
for rxn in glycocalix_rxns_transport:
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")


No objective coefficients in model. Unclear what should be optimized


Reaction ID: MARMAR82812, Reaction: MAM01524e --> MAM01524y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82813, Reaction: MAM01524a --> MAM01524y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82814, Reaction: MAM01522e --> MAM01522y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82815, Reaction: MAM01522a --> MAM01522y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82816, Reaction: MAM01529e --> MAM01529y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82817, Reaction: MAM01529a --> MAM01529y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82818, Reaction: MAM01559e --> MAM01559y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82819, Reaction: MAM01559a --> MAM01559y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82837, Reaction: MAM01562e --> MAM01562y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction ID: MARMAR82838, Reaction: MAM01562a --> MAM01562y, Lower Bound: 0.0, Upper Bound: 1000.0
Reaction I

In [2]:
import cobra
from cobra.flux_analysis import flux_variability_analysis
import copy 
# Full model, that already includes demand reactions 
full_model_path = "model_full_THG_with_exchanges.xml"
full_model = cobra.io.read_sbml_model(full_model_path)

# Model with transport reactions
model_transport = cobra.io.read_sbml_model("model_transport_allrxns_glycocalix_cytoskeleton.xml")

#see reactions in glycocalix (y)
glycocalix_rxns_transport = [rxn for rxn in model_transport.reactions if any(met.compartment == "y" for met in rxn.metabolites)]
print("Transport reactions in glycocalix (y):")
for rxn in glycocalix_rxns_transport:
    #change lower bound to-1000 to make the reaction reversible
    rxn.lower_bound = -1000
    print(f"Reaction ID: {rxn.id}, Reaction: {rxn.reaction}, Lower Bound: {rxn.lower_bound}, Upper Bound: {rxn.upper_bound}")
    

mets_to_add = ["MAM01559y", "MAM01562y", "MAM01524y", "MAM01522y", "MAM00193y", "MAM01529y", "MAM02555y", "MAM02630y","MAM01365y"]
sink_rxn_list = []  # List to keep track of added sink reactions
added_transport_reactions = []  # List to keep track of added transport reactions

for met_id in mets_to_add:
    #find the internal reactions associated with the metabolite
    met = full_model.metabolites.get_by_id(met_id)
    internal_rxns = [rxn for rxn in full_model.reactions if met in rxn.metabolites and not rxn.id.startswith("DM_") and not rxn.id.startswith("MARMAR")] #no demand reactions or transport reactions

    tr_added = False  # Flag to check if a transport reaction was added
    #try to add the transport reaction for the metabolite, TR are stored in the glycocalix_rxns_transport list
    for tr in glycocalix_rxns_transport:
        tr_met_ids = [met.id for met in tr.metabolites]  # Get the metabolite IDs in the transport reaction
        if met.id in tr_met_ids:
            print(f"Transport reaction for {met_id} found: {tr.id}")
            # Check if the reaction already exists in the full model
            tr_copy = copy.deepcopy(tr)  # Create a copy of the transport reaction to avoid modifying the original
            try:
                full_model.reactions.get_by_id(tr_copy.id)
                print(f"Transport reaction {tr_copy.id} already exists in the model.")
                continue  # Skip to the next transport reaction if it already exists
            except KeyError:
                print(f"Transport reaction {tr_copy.id} does not exist in the model. Adding it now.")
                # Add the transport reaction to the full model
                full_model.add_reactions([tr_copy])
                sol = full_model.optimize()
                if sol.status != 'optimal':
                    print(f"Warning: Adding {tr_copy.id} made the model infeasible!. Reaction: {tr_copy.reaction}")
                    full_model.remove_reactions([tr_copy])
                    continue # Skip to the next transport reaction if it made the model infeasible

                #do an FVA on the model after adding the transport reaction, only for the metabolites internal reactions
                fva_results_after = flux_variability_analysis(full_model, reaction_list=internal_rxns, fraction_of_optimum=0.95)
                # Print FVA results for the internal reactions after adding the transport reaction
                for rxn_id, row in fva_results_after.iterrows():
                    min_flux = row['minimum']
                    max_flux = row['maximum']
                    print(f"Reaction ID: {rxn_id}, Min Flux: {min_flux}, Max Flux: {max_flux}")

                    if max_flux > 0:
                        print(f"Transport reaction {tr_copy.id} added successfully for metabolite {met_id}.")
                        tr_added = True
                
                if max_flux > 0:
                    added_transport_reactions.append(tr_copy.id)  # Keep track of added transport reactions
                    break #If one transport reaction was added successfully, break the loop to avoid adding multiple transport reactions for the same metabolite
                else:
                    #remove the transport reaction if it was added but did not result in a positive flux
                    full_model.remove_reactions([tr_copy])
                    print(f"Transport reaction {tr_copy.id} did not result in a positive flux for metabolite {met_id}. Removed it from the model.")
    
    if not tr_added:
        # Add instead a sink reaction for the metabolite if no transport reaction was added
        print(f"No transport reaction found for {met_id}. Adding a sink reaction instead.")
        # Create a new sink reaction
        sink_rxn = cobra.Reaction(f"Sink_{met.id}")
        sink_rxn.name = f"Sink {met.name}"
        sink_rxn.subsystem = 'Putative Sink Reaction'
        sink_rxn.lower_bound = -1000   
        sink_rxn.upper_bound = 1000   
        sink_rxn.add_metabolites({met: 1})  # Add the metabolite as a product
        sink_rxn.notes = {'Confidence Level': '4', 'AUTHORS': 'Sink transport reaction automatically generated'}
        sink_rxn.gene_reaction_rule = ''  # No gene association for sink reactions
        # Add the sink reaction to the model
        full_model.add_reactions([sink_rxn])
        sink_rxn_list.append(sink_rxn.id)  # Keep track of added sink reactions
        print(f"Added sink reaction: {sink_rxn.id} for metabolite: {met.name}")

        #remove DM reaction if it exists
        for rxn in met.reactions:
            if rxn.id.startswith("DM_"):
                full_model.remove_reactions([rxn])
                print(f"Removed DM reaction: {rxn.id} for metabolite: {met.name}")


print("Number of added transport reactions:", len(added_transport_reactions))
print("Added transport reactions:", added_transport_reactions)
print("Number of added sink reactions:", len(sink_rxn_list))
print("Added sink reactions:", sink_rxn_list)

glycocalix_rxns_after = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]

fva_results_after = flux_variability_analysis(full_model, reaction_list=glycocalix_rxns_after, fraction_of_optimum=0.95)
# Print FVA results for glycocalix reactions after adding sink reactions    
for rxn_id, row in fva_results_after.iterrows():
    min_flux = row['minimum']
    max_flux = row['maximum']
    print(f"Reaction ID: {rxn_id}, Min Flux: {min_flux}, Max Flux: {max_flux}")




No objective coefficients in model. Unclear what should be optimized


Transport reactions in glycocalix (y):
Reaction ID: MARMAR82812, Reaction: MAM01524e <=> MAM01524y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82813, Reaction: MAM01524a <=> MAM01524y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82814, Reaction: MAM01522e <=> MAM01522y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82815, Reaction: MAM01522a <=> MAM01522y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82816, Reaction: MAM01529e <=> MAM01529y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82817, Reaction: MAM01529a <=> MAM01529y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82818, Reaction: MAM01559e <=> MAM01559y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82819, Reaction: MAM01559a <=> MAM01559y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82837, Reaction: MAM01562e <=> MAM01562y, Lower Bound: -1000, Upper Bound: 1000.0
Reaction ID: MARMAR82838, Reaction: MAM01562a <=> MA

Trying to minimize use of sink and demand rs

In [3]:
import cobra
from cobra.flux_analysis import flux_variability_analysis


# Step 1: Identify all glycocalyx reactions
glycocalix_rxns = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]

# Step 2: Identify sink and demand reactions in the glycocalyx
sink_rxns = [rxn for rxn in glycocalix_rxns if rxn.id.startswith("Sink_")]
demand_rxns = [rxn for rxn in glycocalix_rxns if rxn.id.startswith("DM_")]

# Step 3: Internal (non-sink, non-demand) reactions in glycocalyx
internal_rxns = [rxn for rxn in glycocalix_rxns if not (rxn.id.startswith("Sink_") or rxn.id.startswith("DM_"))]

print("Initial count:")
print(f"sink reactions: {len(sink_rxns)}")
print(f"Demand reactions: {len(demand_rxns)}")
print(f"Internal reactions: {len(internal_rxns)}")

# Step 4: Try removing each sink/demand reaction one at a time and check FVA
def test_and_remove_rxns(model, rxns_to_test, internal_rxns):
    removed_rxns = []
    for rxn in rxns_to_test:
        print(f"Testing removal of {rxn.id} ...")
        # Remove the reaction
        model.remove_reactions([rxn])
        # Check FVA for internal reactions
        fva = flux_variability_analysis(model, reaction_list=internal_rxns, fraction_of_optimum=0.95)
        blocked = fva['maximum'] <= 0
        if blocked.any():
            print(f"Removal of {rxn.id} blocks {blocked.sum()} internal reactions. Restoring.")
            # If any internal reaction is blocked, restore the reaction
            model.add_reactions([rxn])
        else:
            print(f"Removal of {rxn.id} succeeded! No internal reaction blocked.")
            removed_rxns.append(rxn.id)
    return removed_rxns

# Remove sink reactions if possible
removed_sink = test_and_remove_rxns(full_model, sink_rxns, internal_rxns)

# Re-identify demand reactions after sink removals (as reactions lists may have changed)
demand_rxns = [rxn for rxn in full_model.reactions if rxn.id.startswith("DM_") and any(met.compartment == "y" for met in rxn.metabolites)]
removed_demand = test_and_remove_rxns(full_model, demand_rxns, internal_rxns)

print("sink reactions that could be removed:", removed_sink)
print("Demand reactions that could be removed:", removed_demand)

# Final FVA check
final_fva = flux_variability_analysis(full_model, reaction_list=internal_rxns, fraction_of_optimum=0.95)
blocked_final = final_fva['maximum'] <= 0
print(f"Number of glycocalyx internal reactions still blocked: {blocked_final.sum()}")


Initial count:
sink reactions: 9
Demand reactions: 16
Internal reactions: 10
Testing removal of Sink_MAM01559y ...
Removal of Sink_MAM01559y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM01562y ...
Removal of Sink_MAM01562y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM01524y ...
Removal of Sink_MAM01524y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM01522y ...
Removal of Sink_MAM01522y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM00193y ...
Removal of Sink_MAM00193y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM01529y ...
Removal of Sink_MAM01529y blocks 1 internal reactions. Restoring.
Testing removal of Sink_MAM02555y ...
Removal of Sink_MAM02555y blocks 8 internal reactions. Restoring.
Testing removal of Sink_MAM02630y ...
Removal of Sink_MAM02630y blocks 8 internal reactions. Restoring.
Testing removal of Sink_MAM01365y ...
Removal of Sink_MAM01365y blocks 8 internal reactions

In [4]:
#save full model with removed sink and demand reactions
cobra.io.write_sbml_model(full_model, "model_full_THG_optimized_sinks_demands.xml")
print("Number of reactions in the optimized model:", len(full_model.reactions))

Number of reactions in the optimized model: 24561


In [1]:
import cobra
model_path = "model_full_THG_optimized_sinks_demands.xml"
full_model = cobra.io.read_sbml_model(model_path)

glycocalix_rxns = [rxn for rxn in full_model.reactions if any(met.compartment == "y" for met in rxn.metabolites)]

sink_rxns = [rxn for rxn in glycocalix_rxns if rxn.id.startswith("Sink_")]
demand_rxns = [rxn for rxn in glycocalix_rxns if rxn.id.startswith("DM_")]

still_present = []

#load protected_rs_endoA.txt and check if the reactions are still in the model
with open("protected_rs_endoA.txt", "r") as file:
    protected_reactions = [line.strip() for line in file if line.strip()]

print("Number of original protected reactions:", len(protected_reactions))

# Check if each protected reaction is in the model
for rxn_id in protected_reactions:
    try:
        rxn = full_model.reactions.get_by_id(rxn_id)
        still_present.append(rxn_id)
    except KeyError:
        print(f"{rxn_id} has been removed from the model.")

print("Number of protected reactions still present in the model:", len(still_present))

for sink_rxn in sink_rxns:
    try:
        full_model.reactions.get_by_id(sink_rxn.id)
        if sink_rxn.id not in still_present:
            still_present.append(sink_rxn.id)
    except KeyError:
        print(f"Sink reaction {sink_rxn.id} has been removed from the model.")

for demand_rxn in demand_rxns:
    try:
        full_model.reactions.get_by_id(demand_rxn.id)
        if demand_rxn.id not in still_present:
            still_present.append(demand_rxn.id)
    except KeyError:
        print(f"Demand reaction {demand_rxn.id} has been removed from the model.")

print("Number of protected reactions still present in the model:", len(still_present))
print("Protected reactions still present in the model:")
for rxn_id in still_present:
    print(rxn_id)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-22
Number of original protected reactions: 397
DM_MAM02039y has been removed from the model.
DM_MAM02555y has been removed from the model.
DM_MAM02040y has been removed from the model.
DM_MAM02497y has been removed from the model.
DM_MAM02630y has been removed from the model.
DM_MAM01365y has been removed from the model.
DM_MAM01524y has been removed from the model.
DM_MAM01522y has been removed from the model.
DM_MAM01529y has been removed from the model.
DM_MAM01562y has been removed from the model.
DM_MAM01559y has been removed from the model.
DM_MAM00193y has been removed from the model.
DM_MAM01371y has been removed from the model.
Number of protected reactions still present in the model: 384
Number of protected reactions still present in the model: 393
Protected reactions still present in the model:
MAR09205
MAR09104
MAR09212
MAR09216
MAR09217
MAR09218
MAR09219
MAR09220
MAR09221
MAR09090
MAR09